# Event Study: US Presidential Inauguration (2025-01-20)

Measures abnormal returns of French, Italian and Spanish stocks around the 2025 US presidential inauguration, using CAPM and Fama-French 3-factor expected-return models. Statistical significance is assessed with Patell and BMP (Boehmer-Musumeci-Poulsen) tests, including the Kolari-Pynnonen cross-correlation adjustment. Results are aggregated by GICS sector and by ESG (green/brown) classification.

In [ ]:
import pandas as pd
import pandas_datareader.data as reader

import statsmodels.api as sm
from scipy.stats import ttest_1samp
from scipy.stats import t, norm
import math
import matplotlib.pyplot as plt

import eikon as ek
import os
ek.set_app_key(os.environ["EIKON_APP_KEY"])  # set your own Refinitiv Eikon app key

import yfinance as yf
from datetime import timedelta

In [ ]:
#Data acquisition
stock_returns = pd.read_excel('IT_stock_returns.xlsx', index_col=0, parse_dates=True)
stock_returns = stock_returns.join(pd.read_excel('ES_stock_returns.xlsx', index_col=0, parse_dates=True), how='inner')
stock_returns = stock_returns.join(pd.read_excel('FR_stock_returns.xlsx', index_col=0, parse_dates=True), how='inner')

stock_returns = stock_returns / 100

start_date = stock_returns.index.min().strftime('%Y-%m-%d')
end_date = stock_returns.index.max().strftime('%Y-%m-%d')

market = ek.get_timeseries('.STOXX', fields='CLOSE', start_date=start_date, end_date=end_date, interval='daily')
market['Market'] = market['CLOSE'].pct_change()
market = market[['Market']].dropna()
market = market.sort_index()

stock_attributes_IT = pd.read_excel('IT_stock_attributes.xlsx', index_col=0)
stock_attributes_ES = pd.read_excel('ES_stock_attributes.xlsx', index_col=0)
stock_attributes_FR = pd.read_excel('FR_stock_attributes.xlsx', index_col=0)

stock_attributes = pd.concat([stock_attributes_IT, stock_attributes_ES, stock_attributes_FR])

stock_attributes.columns = ['Industry', 'ESG Score']  # rename for clarity

def classify_green_brown(grade):
    if grade in ['A+', 'A', 'A-']:
        return 'Green'
    elif grade in ['D+', 'D', 'D-']:
        return 'Brown'
    else:
        return 'Unknown/Intermediate'  # optionally use this category

# Apply classification
stock_attributes['Environmental Category'] = stock_attributes['ESG Score'].apply(classify_green_brown)


factors = pd.read_csv('famafrench.csv')
factors['Date'] = pd.to_datetime(factors['Date'], format='%Y%m%d')
factors.set_index('Date', inplace=True)
factors = factors/100

In [ ]:
#Data clean-up
stock_returns = stock_returns.loc[:, stock_returns.isna().mean() < 0.2]
stock_returns = stock_returns.loc[:, (stock_returns == 0).mean() < 0.2]
stock_returns = stock_returns.loc[:, stock_returns.nunique(dropna=True) > 1]
stock_returns = stock_returns.loc[:, (stock_returns.abs() <= 1).all()]
stock_returns = stock_returns.fillna(0)
stock_returns = stock_returns.sort_index()

market = market[['Market']].dropna()


In [ ]:
#COMBINE STOCK WITH MARKET RETURNS AND DEFINE THE WINDOWS
combined_returns = stock_returns.join(market, how='inner')
combined_returns.index = pd.to_datetime(combined_returns.index)


event_date = pd.to_datetime("2025-01-20")
window = 5  # days before and after

# Get list of dates to exclude
event_dates = pd.date_range(event_date - timedelta(days=7), event_date + timedelta(days=7))


estimation_data = combined_returns[(~combined_returns.index.isin(event_dates)) & ~(combined_returns.index >= event_date + timedelta(days=window))]
event_data = combined_returns[combined_returns.index.isin(event_dates)]
event_data = event_data.loc[:, event_data.nunique(dropna=True) > 1]

event_uniques = event_data.nunique()

# Find stocks with more than 1 unique value in the estimation window
estimation_uniques = estimation_data.nunique()

# Keep only stocks that are not flat in the event window or are flat in both
valid_stocks = [stock for stock in event_data.columns if not (event_uniques[stock] == 1 and estimation_uniques[stock] > 1)]


event_data = event_data[valid_stocks]
estimation_data = estimation_data[valid_stocks]

In [ ]:
#OLS Regression using CAPM

capm_df = []

for stock in estimation_data.columns.drop('Market'):
    y = estimation_data[stock]
    X = sm.add_constant(estimation_data['Market'])
    model = sm.OLS(y, X.astype(float)).fit()
    
    capm_df.append({
        'Stock': stock,
        'Alpha': model.params['const'],
        'Beta': model.params['Market'],
        'R-squared': model.rsquared,
        'p-value (Beta)': model.pvalues['Market']
    })

capm_df = pd.DataFrame(capm_df)
capm_df = capm_df.set_index('Stock')

In [ ]:
#Expected return calculation for CAPM

abnormal_returns_event = pd.DataFrame(index=event_data.index)

for index, row in capm_df.iterrows():
    alpha = row['Alpha']
    beta = row['Beta']
    
    abnormal_returns_event[index] = event_data[index] - (alpha + beta * event_data['Market'])

abnormal_returns_estimation = pd.DataFrame(index=estimation_data.index)

for index, row in capm_df.iterrows():
    alpha = row['Alpha']
    beta = row['Beta']
    
    abnormal_returns_estimation[index] = estimation_data[index] - (alpha + beta * estimation_data['Market'])




In [ ]:
# Combine stock returns with Fama-French factors
combined_returns = stock_returns.join(factors, how='inner')
combined_returns.index = pd.to_datetime(combined_returns.index)

# Define event and estimation windows
event_date = pd.to_datetime("2025-01-20")
window = 5
event_dates = pd.date_range(event_date - timedelta(days=7), event_date + timedelta(days=7))

# Separate data
estimation_data = combined_returns[(~combined_returns.index.isin(event_dates)) & ~(combined_returns.index >= event_date + timedelta(days=window))]
event_data = combined_returns[combined_returns.index.isin(event_dates)]

# Drop flat stocks, same as in your CAPM
event_uniques = event_data.nunique()
estimation_uniques = estimation_data.nunique()
valid_stocks = [stock for stock in stock_returns.columns if not (event_uniques[stock] == 1 and estimation_uniques[stock] > 1)]

# Keep valid columns only
estimation_data = estimation_data[valid_stocks + ['Mkt-RF', 'SMB', 'HML', 'RF']]
event_data = event_data[valid_stocks + ['Mkt-RF', 'SMB', 'HML', 'RF']]

# Initialize output DataFrames
abnormal_returns_estimation = pd.DataFrame(index=estimation_data.index)
abnormal_returns_event = pd.DataFrame(index=event_data.index)

# Loop over each valid stock
# Use dictionaries to store series
estimation_ar_dict = {}
event_ar_dict = {}

for stock in valid_stocks:
    est = estimation_data[[stock, 'Mkt-RF', 'SMB', 'HML', 'RF']].dropna()
    y_est = est[stock] - est['RF']
    X_est = sm.add_constant(est[['Mkt-RF', 'SMB', 'HML']])

    model = sm.OLS(y_est, X_est).fit()
    alpha = model.params['const']
    beta_mkt = model.params['Mkt-RF']
    beta_smb = model.params['SMB']
    beta_hml = model.params['HML']

    # Estimation window AR
    est_expected = alpha + beta_mkt * est['Mkt-RF'] + beta_smb * est['SMB'] + beta_hml * est['HML']
    est_ar = y_est - est_expected
    estimation_ar_dict[stock] = est_ar

    # Event window AR
    evt = event_data[[stock, 'Mkt-RF', 'SMB', 'HML', 'RF']].dropna()
    y_evt = evt[stock] - evt['RF']
    evt_expected = alpha + beta_mkt * evt['Mkt-RF'] + beta_smb * evt['SMB'] + beta_hml * evt['HML']
    evt_ar = y_evt - evt_expected
    event_ar_dict[stock] = evt_ar

    #print(f"{stock} : Beta - {beta_mkt:.4f}, R² - {model.rsquared:.4f}")

# Convert to DataFrames once
abnormal_returns_estimation = pd.DataFrame(estimation_ar_dict)
abnormal_returns_event = pd.DataFrame(event_ar_dict)

    
# Done: both abnormal_returns_estimation and abnormal_returns_event are ready


In [ ]:
#ADD ESG SECTOR 
#Comment the Idustry sector part and uncomment this one in order to change the classification 
'''

ticker_to_sector = stock_attributes['Environmental Category'].to_dict()

# Create MultiIndex for columns: (Sector, Ticker)
new_columns = [
    (ticker_to_sector.get(ticker), ticker)
    for ticker in abnormal_returns_estimation.columns
]

# Set the new MultiIndex
abnormal_returns_estimation.columns = pd.MultiIndex.from_tuples(new_columns, names=['Sector', 'Ticker'])
abnormal_returns_event.columns = pd.MultiIndex.from_tuples(new_columns, names=['Sector', 'Ticker'])'''

In [ ]:
#ADD INDUSTRY SECTOR
ticker_to_sector = stock_attributes['Industry'].to_dict()

# Create MultiIndex for columns: (Sector, Ticker)
new_columns = [
    (ticker_to_sector.get(ticker, 'Unknown'), ticker)
    for ticker in abnormal_returns_estimation.columns
]
new_columns_s = [
    (ticker_to_sector.get(ticker, 'Unknown'), ticker)
    for ticker in stock_returns.columns
]
# Set the new MultiIndex
abnormal_returns_estimation.columns = pd.MultiIndex.from_tuples(new_columns, names=['Sector', 'Ticker'])
abnormal_returns_event.columns = pd.MultiIndex.from_tuples(new_columns, names=['Sector', 'Ticker'])

In [ ]:
#Patell test

def run_patell_test(abnormal_returns_estimation, abnormal_returns_event, estimation_data, event_data, event_date, market_column):
    # Compute mean market return in estimation window
    market_mean = estimation_data[market_column].mean()
    M_i = abnormal_returns_estimation.count(axis=0)
    std_AR = ((abnormal_returns_estimation**2).sum(axis=0)/(M_i-2))**0.5
    # Standard error matrix for ARs on each event day
    std_AR_i = pd.DataFrame(index=event_data.index, columns=abnormal_returns_estimation.columns)

    for date in event_data.index:
        std_AR_i.loc[date] = std_AR * (
            1
            + 1 / M_i
            + ((event_data[market_column] - market_mean) ** 2).loc[date] /
            ((estimation_data[market_column] - market_mean) ** 2).sum()
        ) ** 0.5

    # Standardized Abnormal Returns
    SAR = abnormal_returns_event / std_AR_i

    # Cumulative and single-day standardized ARs
    CSAR = SAR.sum()
    ASAR = SAR.loc[event_date].sum()

    # Standard deviations
    std_ASAR = (((M_i - 2) / (M_i - 4)).sum()) ** 0.5
    std_CSAR = (len(abnormal_returns_event) * ((M_i - 2) / (M_i - 4))) ** 0.5

    # Patell Z-statistics (uncorrected)
    z_patell_CAAR = CSAR.count() ** (-0.5) * ((CSAR / std_CSAR).sum())
    z_patell_AAR = ASAR / std_ASAR

    # Corresponding p-values
    p_patell_CAAR = 2 * (1 - norm.cdf(abs(z_patell_CAAR)))
    p_patell_AAR = 2 * (1 - norm.cdf(abs(z_patell_AAR)))

    # Kolari-Pynnönen correlation correction
    corr_matrix = abnormal_returns_estimation.corr()
    N = len(corr_matrix)
    r_bar = (corr_matrix.values.sum() - N) / (N * (N - 1))

    z_adj_patell_CAAR = z_patell_CAAR * (((1 - r_bar) / (1 + (N - 1) * r_bar)) ** 0.5)
    z_adj_patell_AAR = z_patell_AAR * (((1 - r_bar) / (1 + (N - 1) * r_bar)) ** 0.5)

    p_adj_patell_CAAR = 2 * (1 - norm.cdf(abs(z_adj_patell_CAAR)))
    p_adj_patell_AAR = 2 * (1 - norm.cdf(abs(z_adj_patell_AAR)))
    
    '''results = {
        'Test': ['Patell (CAAR)', 'Adj Patell (CAAR)', f'Patell (AAR) {event_date}', f'Adj Patell (AAR) {event_date}'],
        #'z-statistic': [z_patell_CAAR, z_adj_patell_CAAR, z_patell_AAR, z_adj_patell_AAR],
        'p-value': [p_patell_CAAR, p_adj_patell_CAAR, p_patell_AAR, p_adj_patell_AAR]
    }
    
    
    results = {
        'Test': [f'Patell (AAR) {event_date}', f'Adj Patell (AAR) {event_date}'],
        #'z-statistic': [z_patell_CAAR, z_adj_patell_CAAR, z_patell_AAR, z_adj_patell_AAR],
        'p-value': [p_patell_AAR, p_adj_patell_AAR]
    }
    '''
    results = {
        'Test': ['Patell (CAAR)', 'Adj Patell (CAAR)'],
        #'z-statistic': [z_patell_CAAR, z_adj_patell_CAAR, z_patell_AAR, z_adj_patell_AAR],
        'p-value': [p_patell_CAAR, p_adj_patell_CAAR]
    }
    
    return pd.DataFrame(results).set_index('Test')



In [ ]:
def run_bmp_test(abnormal_returns_estimation, abnormal_returns_event, estimation_data, event_data, market_column, event_date):
    """
    Computes the BMP and adjusted BMP test statistics and p-values for cumulative abnormal returns (CAR).

    Parameters:
    - abnormal_returns_estimation: pd.DataFrame of ARs in estimation window (rows: dates, cols: firms)
    - abnormal_returns_event: pd.DataFrame of ARs in event window (same structure)
    - estimation_data: pd.DataFrame containing market returns during estimation window (with market_col)
    - event_data: pd.DataFrame containing market returns during event window (with market_col)
    - market_col: str, name of the market return column (default 'Market')

    Returns:
    - dict with BMP t-statistic, BMP p-value, adjusted BMP t-statistic, adjusted BMP p-value
    """

    market_mean = estimation_data[market_column].mean()
    M_i = abnormal_returns_estimation.count(axis=0)
    # Correlation matrix of estimation window ARs across firms
    corr_matrix = abnormal_returns_estimation.corr()

    # Number of firms
    N = len(corr_matrix)

    # Average correlation (excluding diagonal)
    r_bar = (corr_matrix.values.sum() - N) / (N * (N - 1))


    std_AR_i = pd.DataFrame(index=event_data.index, columns=abnormal_returns_estimation.columns)
    std_AR = ((abnormal_returns_estimation**2).sum(axis=0)/(M_i-2))**0.5
    # Loop over event days to fill in the standard error
    for date in event_data.index:
        std_AR_i.loc[date] = std_AR * (
        1
        + 1 / M_i
        + ((event_data[market_column] - market_mean) ** 2).loc[date] / ((estimation_data[market_column] - market_mean) ** 2).sum()
        ) ** 0.5
    
    SAR = abnormal_returns_event / std_AR_i
    
    ASAR = SAR.loc[event_date].sum()
    
    std_ASAR = SAR.loc[event_date].std()

    t_BMP_AAR = ASAR/(std_ASAR*(N**0.5))

    t_BMP_AAR_adj = t_BMP_AAR * ((1 - r_bar) / (1 + (N - 1) * r_bar)) ** 0.5

    p_BMP_AAR = 2 * (1 - t.cdf(abs(t_BMP_AAR), df=M_i.mean()-1))
    p_BMP_AAR_adj = 2 * (1 - t.cdf(abs(t_BMP_AAR_adj), df=M_i.mean()-1))

    # Calculate CAR (sum of abnormal returns in event window) per firm
    CAR = abnormal_returns_event.sum()
    
    # Calculate standard deviation for CAR using estimation data and market variance
    std_CAR = abnormal_returns_estimation.std() * (
        abnormal_returns_event.count()
        + abnormal_returns_event.count() / abnormal_returns_estimation.count()
        + ((event_data[market_column] - market_mean) ** 2).sum() / ((estimation_data[market_column] - market_mean) ** 2).sum()
    ) ** 0.5

    # Standardized cumulative abnormal returns (SCAR)
    SCAR = CAR / std_CAR

    # BMP t-statistic (mean SCAR scaled by its sample std and sample size)
    t_BMP = (SCAR.mean() / SCAR.std()) * (SCAR.count() ** 0.5)

    # Adjust BMP t-statistic for cross-sectional correlation
    t_BMP_adj = t_BMP * ((1 - r_bar) / (1 + (N - 1) * r_bar)) ** 0.5

    # p-values for BMP and adjusted BMP (two-tailed t-test)
    df = M_i.mean() - 1
    p_BMP = 2 * (1 - t.cdf(abs(t_BMP), df=df))
    p_BMP_adj = 2 * (1 - t.cdf(abs(t_BMP_adj), df=df))
    '''
    results = {
        'Test': ['BMP (CAAR)', 'Adj BMP (CAAR)', f'BMP (AAR) {event_date}', f'Adj BMP (AAR) {event_date}'],
    #   't-statistic': [t_BMP, t_BMP_adj, t_BMP_AAR, t_BMP_AAR_adj],
        'p-value': [p_BMP, p_BMP_adj, p_BMP_AAR, p_BMP_AAR_adj]
    }
    results = {
        'Test': [f'BMP (AAR) {event_date}', f'Adj BMP (AAR) {event_date}'],
    #   't-statistic': [t_BMP, t_BMP_adj, t_BMP_AAR, t_BMP_AAR_adj],
        'p-value': [p_BMP_AAR, p_BMP_AAR_adj]
    }
    
    '''
    results = {
        'Test': ['BMP (CAAR)', 'Adj BMP (CAAR)'],
    #   't-statistic': [t_BMP, t_BMP_adj, t_BMP_AAR, t_BMP_AAR_adj],
        'p-value': [p_BMP, p_BMP_adj]
    }
    
    return pd.DataFrame(results).set_index('Test')


In [ ]:
results = []  # list of DataFrames to concatenate later
event_date = '2025-01-20'

for sector in abnormal_returns_estimation.columns.levels[0]:
    print(f"CAAR {sector}: {abnormal_returns_event[sector].sum().mean()}")
    print(f"AAR: {abnormal_returns_event[sector].loc[event_date].mean()}")

    # Run tests
    patell_df = run_patell_test(
        abnormal_returns_estimation[sector],
        abnormal_returns_event[sector],
        estimation_data,
        event_data,
        event_date,
        'Mkt-RF'
    )
    bmp_df = run_bmp_test(
        abnormal_returns_estimation[sector],
        abnormal_returns_event[sector],
        estimation_data,
        event_data,
        'Mkt-RF',
        event_date
    )
    # Add sector name as a column
    patell_df = patell_df.reset_index()
    patell_df['Sector'] = sector

    bmp_df = bmp_df.reset_index()
    bmp_df['Sector'] = sector

    # Combine both tests for this sector
    combined = pd.concat([patell_df, bmp_df], ignore_index=True)
    results.append(combined)

# Combine all sectors into one DataFrame
final_results = pd.concat(results, ignore_index=True)

# Optional: reorder columns
final_results = final_results[['Sector', 'Test', 'p-value']]

# Display the result
print(final_results)

In [ ]:
# Define category mappings

# Ensure event_date is a Timestamp
event_date = pd.to_datetime(event_date)

plt.figure(figsize=(12, 6))

# Use the shared date index (assuming all sectors share the same one)
date_index = abnormal_returns_event.index.sort_values()
if event_date not in date_index:
    raise ValueError("Event date not found in index!")

# Create a mapping from date to relative trading day (0 at event date)
event_position = date_index.get_loc(event_date)
relative_days = pd.Series(data=range(-event_position, len(date_index) - event_position), index=date_index)

# Now plot for each sector
for sector in abnormal_returns_event.columns.levels[0]:
    sector_data = abnormal_returns_event[sector]

    aar = sector_data.mean(axis=1).sort_index().fillna(0)
    caar = aar.cumsum()

    # Plot with trading-day-relative x-axis
    x_vals = relative_days.loc[caar.index]
    plt.plot(x_vals, caar.values * 100,
             label=f'{sector}')

# Add event line
plt.axvline(x=0, color='red', linestyle=':', label='Event Day')

# Formatting
plt.title('Cumulative Average Abnormal Returns (CAAR) by Sector')
plt.xlabel('Trading Days Relative to Event')
plt.ylabel('CAAR (%)')
plt.legend(title='Sector (Category)', bbox_to_anchor=(1.05, 1), loc='upper left')
plt.grid(True)
plt.tight_layout()
plt.show()


In [ ]:
#ESG AARs figure

import matplotlib.pyplot as plt
import seaborn as sns
import pandas as pd

stock_columns = abnormal_returns_event.columns

# Create a DataFrame: rows = date, columns = stock, values = abnormal return
ab_returns = abnormal_returns_event.copy()
ab_returns.columns = stock_columns

# Melt into long format: date, stock, abnormal return
ab_returns_long = ab_returns.reset_index().melt(id_vars='Date', var_name='Stock', value_name='Abnormal Return')
ab_returns_long = ab_returns_long.rename(columns={'index': 'Date'})

# Add environmental category
ab_returns_long = ab_returns_long.merge(
    stock_attributes[['Environmental Category']],
    left_on='Stock',
    right_index=True,
    how='left'
)

# Drop any rows with missing category or return
ab_returns_long = ab_returns_long.dropna(subset=['Environmental Category', 'Abnormal Return'])

# Convert Date to string without time part (just YYYY-MM-DD)
ab_returns_long['Date'] = pd.to_datetime(ab_returns_long['Date'])
ab_returns_long['Date'] = ab_returns_long['Date'].dt.strftime('%Y-%m-%d')

# Make it categorical in sorted order
ab_returns_long['Date'] = pd.Categorical(
    ab_returns_long['Date'],
    categories=sorted(ab_returns_long['Date'].unique()),
    ordered=True
)

# Setup categories and palette
categories = ['Green', 'Brown', 'Unknown/Intermediate']
palette = {'Green': 'green', 'Brown': 'saddlebrown', 'Unknown/Intermediate': 'steelblue'}

# Prepare figure with 3 subplots, side-by-side
fig, axes = plt.subplots(nrows=1, ncols=3, figsize=(15, 6), sharey=True)

for ax, category in zip(axes, categories):
    # Filter data for this category
    data_cat = ab_returns_long[ab_returns_long['Environmental Category'] == category]

    sns.boxplot(
        data=data_cat,
        x='Date',
        y='Abnormal Return',
        color=palette[category],
        ax=ax,
        showfliers=False
    )
    ax.set_title(f'Daily Abnormal Returns for {category} Stocks')
    ax.set_xlabel('Trading Day')
    ax.set_ylabel('Abnormal Return')
    ax.grid(True)
    ax.tick_params(axis='x', rotation=45)

    # Fix x-ticks alignment with shortened labels
    ticks = range(len(data_cat['Date'].cat.categories))
    ax.set_xticks(ticks)
    ax.set_xticklabels(data_cat['Date'].cat.categories, rotation=45, ha='right')

plt.tight_layout()
plt.show()


In [ ]:
import pandas as pd
import numpy as np
from scipy.stats import skew, kurtosis

# 1. Filter for French tickers
french_returns = (stock_returns*100).loc[:, stock_returns.columns.str.endswith('.PA')]
# 2. Merge sector info
stock_sectors = stock_attributes.loc[stock_attributes.index.str.endswith('.PA'), 'Industry']
french_returns = french_returns.loc[:, french_returns.columns.isin(stock_sectors.index)]

# 3. Melt to long format
french_long = french_returns.reset_index().melt(id_vars='Date', var_name='Stock', value_name='Return')
french_long['Sector'] = french_long['Stock'].map(stock_sectors)

# 4. Group by Sector
def describe_group(group):
    returns = group['Return']
    return pd.Series({
        'Mean': returns.mean(),
        'Std. Dev': returns.std(),
        'Min': returns.min(),
        'Q1': returns.quantile(0.25),
        'Median': returns.median(),
        'Q3': returns.quantile(0.75),
        'Max': returns.max(),
        'Skewness': skew(returns, nan_policy='omit'),
        'Kurtosis': kurtosis(returns, nan_policy='omit', fisher=False)  # Use Pearson's definition
    })

summary_table = french_long.groupby('Sector').apply(describe_group).round(2)
print(summary_table)
